In [1]:
import os
%pwd

'd:\\Github Projects\\NLP MLOPs Project\\research'

In [2]:
os.chdir('../')

In [3]:
%pwd

'd:\\Github Projects\\NLP MLOPs Project'

Config

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: str
    data_path: Path
    tokenizer_name: Path


Configuration Manager

In [5]:
from src.textSummarizer.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.textSummarizer.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManger:
    def __init__(self,
                    config_filepath = CONFIG_FILE_PATH,
                    params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self)-> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            tokenizer_name = config.tokenizer_name
        )

        return data_transformation_config

Componants

In [7]:
import os
from src.textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_from_disk

d:\Github Projects\NLP MLOPs Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class DataTransformation:
    def __init__(self, config=DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):
        # Encode the dialogue (input)
        input_encodings = self.tokenizer(
            example_batch["dialogue"],
            max_length=1024,
            truncation=True
        )

        # Encode the summary (target/label)
        target_encodings = self.tokenizer(
            text_target=example_batch["summary"],
            max_length=128,
            truncation=True
        )

        return {
            "input_ids": input_encodings["input_ids"],
            "attention_mask": input_encodings["attention_mask"],
            "labels": target_encodings["input_ids"]
        }

    def convert(self):
        data_samsum = load_from_disk(self.config.data_path)
        data_samsum_pt = data_samsum.map(self.convert_examples_to_features, batched=True)
        data_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset"))
        



### Configuratoin Manager

In [9]:
config = ConfigurationManger()
data_transformation_config = config.get_data_transformation_config()
data_transformation = DataTransformation(config=data_transformation_config)
data_transformation.convert()

[2026-09-08 23:51:10,988: INFO : common : yaml file: config\config.yaml loaded successfully]
[2026-09-08 23:51:10,990: INFO : common : yaml file: params.yaml loaded successfully]
[2026-09-08 23:51:10,992: INFO : common : created directory at: artifacts]
[2026-09-08 23:51:10,994: INFO : common : created directory at: artifacts/data_transformation]
[2026-09-08 23:51:11,572: INFO : _client : HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"]
[2026-09-08 23:51:11,971: INFO : _client : HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]


[2026-09-08 23:51:11,973: WARNING : _http : Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-09-08 23:51:11,986: INFO : _client : HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-09-08 23:51:12,013: INFO : _client : HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-09-08 23:51:12,311: INFO : _client : HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-09-08 23:51:12,323: INFO : _client : HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 63194.22 examples/s]
